In [4]:
# !pip install langchain google-generativeai
# !pip install langchain-google-genai
# !pip install -U langchain-huggingface sentence-transformers langchain-chroma chromadb
# !pip install pdf2image pytesseract
# !pip install "unstructured[pdf]"

In [5]:
import os
import time
from dotenv import load_dotenv
load_dotenv()  # reads GOOGLE_API_KEY from .env

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.documents import Document
from pydantic import BaseModel, Field

In [10]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1684.41it/s]


In [16]:
vector_store= Chroma(embedding_function=embeddings, persist_directory="./chroma_db")

In [17]:
import pandas as pd

In [18]:
df = pd.read_csv(r"C:\Users\aspir\Desktop\rag_practice\Rag_project\housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [21]:
df_new= df.drop(["longitude", "latitude"], axis=1)

In [22]:
df_new["summary"] = (
    "the person of age " + df["housing_median_age"].astype(str) +
    " total bedrooms " + df["total_bedrooms"].astype(str) +
    " of median house value "+ df['median_house_value'].astype(str)+ " stays " + df["ocean_proximity"].astype(str)
)

In [23]:
documents = []

for i in range(0, len(df_new), 20):
    batch = df_new.iloc[i:i+20]

    text = " ".join(
        f"Row {idx}: {row['summary']}"
        for idx, row in batch.iterrows()
    )

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": "housing.csv",
                "start_row": int(batch.index[0]),
                "end_row": int(batch.index[-1])
            }
        )
    )

In [24]:
for i, doc in enumerate(documents):
  vector_store.add_documents([doc], ids=[str(i)])

In [25]:
results = vector_store.similarity_search_with_score(query="what is the median age of the persons in bay area ", k=3)


In [26]:
# for doc, score in results:
#     print(f"* [SIM={score:3f}] {doc.page_content} [{doc.metadata}]")
doc, score= results[0]
print(score)

0.7968829870223999


In [27]:
import fitz
import os
from PIL import Image
import io
import pytesseract

def pdf_to_page_images(pdf_path, output_dir="pdf_pages", dpi=250):
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    pages = []

    for i, page in enumerate(doc):
        pix = page.get_pixmap(
            matrix=fitz.Matrix(dpi / 72, dpi / 72),
            alpha=False
        )

        image_path = os.path.join(
            output_dir, f"page_{i + 1}.png"
        )

        pix.save(image_path)

        pages.append({
            "page": i + 1,
            "image_path": image_path
        })

    doc.close()
    return pages



def extract_ocr_text(image_path):
    image = Image.open(image_path).convert("RGB")

    text = pytesseract.image_to_string(
        image,
        config="--psm 3"
    )

    return text.strip()

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter